In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [1]:
#Step2
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-smallset-finalF1_Numbered_NoNullStr.parquet")

In [2]:
#Step3
balanced_df = Epilepsy_Combined.drop("personid")

In [4]:
#Step4
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
import random
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, VectorSizeHint

# Step 0: Mimicking randomSplit with seed and randomness
def probabilistic_split(df: DataFrame, fractions: list, seed=None) -> list:
    if seed is not None:
        random.seed(seed)

    cumulative_fractions = [sum(fractions[:i + 1]) for i in range(len(fractions))]
    random_col = F.rand(seed)
    df_with_random = df.withColumn("random", random_col)
    splits = []
    prev_fraction = 0
    for fraction in cumulative_fractions:
        split_df = df_with_random.filter((F.col("random") >= prev_fraction) & (F.col("random") < fraction))
        splits.append(split_df.drop("random"))
        prev_fraction = fraction
    return splits

# Train, validation, and test sampling fractions
train_fraction = 0.7
valid_fraction = 0.2
test_fraction = 0.1
fractions = [train_fraction, valid_fraction, test_fraction]
seed_value = 23
train_data, valid_data, test_data = probabilistic_split(balanced_df, fractions, seed=seed_value)

# Show class distribution in train, validation, and test datasets
train_data.groupBy('label').count().show()
valid_data.groupBy('label').count().show()
test_data.groupBy('label').count().show()

# Print counts
print("Sampled data count:", balanced_df.count())
print("Train data count:", train_data.count())
print("Validation data count:", valid_data.count())
print("Test data count:", test_data.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-----+-----+
|label|count|
+-----+-----+
|  0.0|70345|
|  1.0|10738|
+-----+-----+



...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Test data count: 11792


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
#Step5
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

▸,:,


In [6]:
#Step6
###########################Included Standard Scalar #############################################################
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint, StandardScaler

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and return size hint stages for the pipeline
def get_vector_size_hint_stage(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 0.01% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")
        # Return a VectorSizeHint stage for the pipeline if vector size is valid
        if vector_size > 0:
            return VectorSizeHint(inputCol=col_name, size=vector_size)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")
    
    return None

# Step 3: Create a list of VectorSizeHint stages for each vector column
vector_size_hint_stages = []
for col_name in vector_cols:
    print(f"Getting VectorSizeHint for vector column: '{col_name}'")
    size_hint_stage = get_vector_size_hint_stage(train_data, col_name)
    if size_hint_stage:
        vector_size_hint_stages.append(size_hint_stage)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")

# Step 6: Initialize StandardScaler
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Step 7: Create a pipeline with VectorSizeHint stages, VectorAssembler, and StandardScaler
pipeline_stages = vector_size_hint_stages + [final_assembler, standardScaler]
pipeline_final = Pipeline(stages=pipeline_stages)

# Step 8: Fit the pipeline on train_data
model_final = pipeline_final.fit(train_data)
print("Pipeline fitting done.")

# Step 9: Transform train, valid, and test datasets using the fitted pipeline
train_data = model_final.transform(train_data)
valid_data = model_final.transform(valid_data)
test_data = model_final.transform(test_data)
print("Pipeline transformation done.")

▸,:,


Getting VectorSizeHint for vector column: 'gender_onehot'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Pipeline fitting done.


<IPython.core.display.Javascript object>

Pipeline transformation done.


In [7]:
#Step7
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

▸,:,


In [8]:
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
# Initialize the GBTClassifier
gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled")
# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for max_depth in paramGrid["maxDepth"]:
    for max_iter in paramGrid["maxIter"]:
        # Set hyperparameters
        gbt.setMaxDepth(max_depth)
        gbt.setMaxIter(max_iter)

        # Fit the model on the training data
        model = gbt.fit(train_data)
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxDepth: {max_depth}, MaxIter: {max_iter}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxDepth": max_depth, "maxIter": max_iter}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")
# # Cast the label column in the test data to double (if necessary)
# test_data = test_data.withColumn('label', test_data.label.cast('double'))
# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10
Train AUC: 0.8662752117814997, Train Accuracy: 0.9187745890013936
Validation AUC: 0.8646186240295742, Validation Accuracy: 0.9206793981980429
MaxDepth: 5, MaxIter: 20
Train AUC: 0.887827485060493, Train Accuracy: 0.9306266418361432
Validation AUC: 0.8843907251358658, Validation Accuracy: 0.9322757253093072
MaxDepth: 10, MaxIter: 10
Train AUC: 0.9150702393907503, Train Accuracy: 0.9425403598781495
Validation AUC: 0.9020006665508653, Validation Accuracy: 0.9348622666724146
MaxDepth: 10, MaxIter: 20
Train AUC: 0.9345003501024468, Train Accuracy: 0.9507418324432988
Validation AUC: 0.9132769535467242, Validation Accuracy: 0.9386127516489201
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.9132769535467242
Best Validation Accuracy: 0.9386127516489201
Test AUC: 0.9411324046312769
Test Accuracy: 0.9355495251017639


In [9]:
###################Using Balanced Samples without using valid set  ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 10180.0         FP: 713.0
Actual: 1     FN: 47.0         TP: 852.0
[[10180.   713.]
 [   47.   852.]]

Metrics:
f1_1:  0.6915584415584416
f1_0:  0.9640151515151515
precision_1:  0.544408945686901
precision_0:  0.9954043218930283
recall_1:  0.9477196885428254
recall_0:  0.9345451207197283
auc:  0.9411324046312769
accuracy:  0.9355495251017639
sensitivity:  0.9477196885428254
specificity:  0.9345451207197283


In [18]:
#Step10
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_data)

            # Validate on validation set
            val_data_pred = model.transform(valid_data)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")

# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")

# Print results
print(f"Train Accuracy: {train_accuracy}, Train AUC: {train_auc}")
print(f"Validation Accuracy: {val_accuracy}, Validation AUC: {val_auc}")
print(f"Test Accuracy: {test_accuracy}, Test AUC: {test_auc}")

Best parameters: maxDepth=10, maxIter=20, Validation AUC=0.9134
Train Accuracy: 0.9507, Train AUC: 0.9348
Validation Accuracy: 0.9386, Validation AUC: 0.9134
Test Accuracy: 0.9355, Test AUC: 0.9113


In [19]:
#Step11
###################To align with MulticlassMetrics, swap the calculations for precision and recall between class 0 and class 1 in the code.##
################################precision_0 and recall_0 now represent the values that would be calculated for class 1###############
#############################precision_1 and recall_1 now represent the values that would be calculated for class 0#####################
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 0 (now swapping with class 1)
    precision_0 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_0 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 1 (now swapping with class 0)
    precision_1 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_1 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "sensitivity": recall_1,  # Sensitivity is recall for class 1
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
print("Confusion Matrix Metrics:")
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

# Print sensitivity separately
print(f"Sensitivity (Recall for Class 1): {metrics['sensitivity']:.4f}")

Confusion Matrix Metrics:
True Positives: 852
True Negatives: 10180
False Positives: 47
False Negatives: 713
Accuracy: 0.9355
Precision 1: 0.9345
Recall 1: 0.9954
Sensitivity: 0.9954
F1 Score 1: 0.9640
Precision 0: 0.9477
Recall 0: 0.5444
F1 Score 0: 0.6916
Specificity: 0.9954
Sensitivity (Recall for Class 1): 0.9954


In [20]:
######################Old Calculation where precision_0 and recall_0: Class 1 and precision_1 and recall_1: Class0 ###################
# Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "sensitivity": recall_1,  # Add sensitivity (recall for class 1)
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
print("Confusion Matrix Metrics:")
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

# Print sensitivity separately
print(f"Sensitivity (Recall for Class 1): {metrics['sensitivity']:.4f}")

Confusion Matrix Metrics:
True Positives: 852
True Negatives: 10180
False Positives: 47
False Negatives: 713
Accuracy: 0.9355
Precision 1: 0.9477
Recall 1: 0.5444
Sensitivity: 0.5444
F1 Score 1: 0.6916
Precision 0: 0.9345
Recall 0: 0.9954
F1 Score 0: 0.9640
Specificity: 0.9954
Sensitivity (Recall for Class 1): 0.5444


In [21]:
#################Class Balance ###################################################
#Step8
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

Number of 0's in the label column: 70345
Number of 1's in the label column: 10738


In [22]:
#Step9
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
count_zeros = 70345
count_ones = 10738

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced= upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame:  10734
Number of 1's in the balanced DataFrame:  70394


In [23]:
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
# Initialize the GBTClassifier
gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled")
# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for max_depth in paramGrid["maxDepth"]:
    for max_iter in paramGrid["maxIter"]:
        # Set hyperparameters
        gbt.setMaxDepth(max_depth)
        gbt.setMaxIter(max_iter)

        # Fit the model on the training data
        model = gbt.fit(train_data_balanced)
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data_balanced)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxDepth: {max_depth}, MaxIter: {max_iter}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxDepth": max_depth, "maxIter": max_iter}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")
# # Cast the label column in the test data to double (if necessary)
# test_data = test_data.withColumn('label', test_data.label.cast('double'))
# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10
Train AUC: 0.8650001150859474, Train Accuracy: 0.8682082634848635
Validation AUC: 0.8524775994860412, Validation Accuracy: 0.1293270681553649
MaxDepth: 5, MaxIter: 20
Train AUC: 0.8810282637163671, Train Accuracy: 0.8684054826940144
Validation AUC: 0.8668298857471979, Validation Accuracy: 0.12993059447342328
MaxDepth: 10, MaxIter: 10
Train AUC: 0.9142545268864094, Train Accuracy: 0.9021915984616902
Validation AUC: 0.860317176816331, Validation Accuracy: 0.501142389102039
MaxDepth: 10, MaxIter: 20
Train AUC: 0.9312237400297604, Train Accuracy: 0.9084779607533774
Validation AUC: 0.8728218538923843, Validation Accuracy: 0.5095486485321378
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.8728218538923843
Best Validation Accuracy: 0.5095486485321378
Test AUC: 0.593174008517018
Test Accuracy: 0.5126356852103121


In [24]:
###################Using Balanced Samples without using valid set  ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 4574.0         FP: 94.0
Actual: 1     FN: 5653.0         TP: 1471.0
[[4574.   94.]
 [5653. 1471.]]

Metrics:
f1_1:  0.3385890206007596
f1_0:  0.6141658274588788
precision_1:  0.9399361022364218
precision_0:  0.4472474821550797
recall_1:  0.20648512071869737
recall_0:  0.9798628963153385
auc:  0.593174008517018
accuracy:  0.5126356852103121
sensitivity:  0.20648512071869737
specificity:  0.9798628963153385


In [8]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Check if we need to upsample or downsample
if count_zeros > count_ones:
    # Upsample the minority class
    upsample_ratio = count_zeros // count_ones  # Calculate integer part of ratio
    remaining_samples_fraction = (count_zeros % count_ones) / count_ones  # Remaining fraction

    # Duplicate the minority class to match the majority class count
    upsampled_minority_class_df = minority_class_df
    for _ in range(upsample_ratio - 1):  # -1 because we already have one instance
        upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)

    # Add the remaining samples to reach exact count
    upsampled_minority_class_df = upsampled_minority_class_df.union(
        minority_class_df.sample(withReplacement=True, fraction=remaining_samples_fraction)
    )
    
    # Combine with majority class
    train_data_balanced = majority_class_df.union(upsampled_minority_class_df)

else:
    # Downsample the majority class if count_ones > count_zeros
    downsample_fraction = count_ones / count_zeros
    downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)
    
    # Combine with minority class
    train_data_balanced = downsampled_majority_class_df.union(minority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 1).count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of 1's in the balanced DataFrame: 70206


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
# Initialize the GBTClassifier
gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled")
# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for max_depth in paramGrid["maxDepth"]:
    for max_iter in paramGrid["maxIter"]:
        # Set hyperparameters
        gbt.setMaxDepth(max_depth)
        gbt.setMaxIter(max_iter)

        # Fit the model on the training data
        model = gbt.fit(train_data_balanced)
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data_balanced)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxDepth: {max_depth}, MaxIter: {max_iter}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxDepth": max_depth, "maxIter": max_iter}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")
# # Cast the label column in the test data to double (if necessary)
# test_data = test_data.withColumn('label', test_data.label.cast('double'))
# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxIter: 10
Train AUC: 0.8788890357638347, Train Accuracy: 0.789154398691416
Validation AUC: 0.8728429557957097, Validation Accuracy: 0.8247187136267621
MaxDepth: 5, MaxIter: 20
Train AUC: 0.8965273595375307, Train Accuracy: 0.8043240167840126
Validation AUC: 0.888250960556312, Validation Accuracy: 0.8295038151485106
MaxDepth: 10, MaxIter: 10
Train AUC: 0.9227905509891097, Train Accuracy: 0.8435388663679682
Validation AUC: 0.891052792989415, Validation Accuracy: 0.8644221235504591
MaxDepth: 10, MaxIter: 20
Train AUC: 0.9379303013984669, Train Accuracy: 0.866787568451746
Validation AUC: 0.9031376261314521, Validation Accuracy: 0.877484157434151
Best Parameters: {'maxDepth': 10, 'maxIter': 20}
Best Validation AUC: 0.9031376261314521
Best Validation Accuracy: 0.877484157434151
Test AUC: 0.7465017926576781
Test Accuracy: 0.8805122116689281


In [33]:
###################Using Balanced Samples without using valid set  ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 9239.0         FP: 421.0
Actual: 1     FN: 988.0         TP: 1144.0
[[9239.  421.]
 [ 988. 1144.]]

Metrics:
f1_1:  0.6188801731133352
f1_0:  0.9291496957811635
precision_1:  0.7309904153354633
precision_0:  0.9033929793683387
recall_1:  0.5365853658536586
recall_0:  0.9564182194616977
auc:  0.7465017926576781
accuracy:  0.8805122116689281
sensitivity:  0.5365853658536586
specificity:  0.9564182194616977


In [ ]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql import DataFrame
import random

# Number of 0's in the label column
count_zeros = 100794
# Number of 1's in the label column
count_ones = 15278

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class
ratio = int(count_zeros / count_ones)
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(ratio - 1):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Combine the upsampled minority class with the original majority class
upsampled_df = upsampled_minority_class_df.union(majority_class_df)

# Downsampling
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(False, count_ones / count_zeros, seed=42)
# Combine the downsampled majority class with the original minority class
downsampled_df = downsampled_majority_class_df.union(minority_class_df)

# Display the counts after balancing
print(f"Upsampled DataFrame: {upsampled_df.groupBy('label').count().collect()}")
print(f"Downsampled DataFrame: {downsampled_df.groupBy('label').count().collect()}")

In [21]:
#############Support Vector Machine (SVM) classifier with an RBF kernel using PySpark ML ##########################################
################# Used LinearSVC class from PySpark since PySpark ML currently does not support RBF kernel SVM directly ############

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import DataFrame

# Define hyperparameter grid for SVM (linear approximation as PySpark does not support RBF kernel directly)
param_grid = {
    "maxIter": [10, 20],
    "regParam": [0.1, 0.01]  # Regularization parameter
}

# Function to perform hyperparameter tuning and model selection
def tune_svm_model(train_data_balanced: DataFrame, valid_data: DataFrame, param_grid: dict, label_col: str):
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    
    best_model = None
    best_auc = 0.0
    best_params = {}

    # Loop through all combinations of hyperparameters
    for max_iter in param_grid["maxIter"]:
        for reg_param in param_grid["regParam"]:
            # Instantiate a new LinearSVC for each hyperparameter combination
            svm = LinearSVC(labelCol=label_col, featuresCol="features_scaled", maxIter=max_iter, regParam=reg_param)

            # Fit the model on the training data
            model = svm.fit(train_data_balanced)

            # Evaluate the model on the validation data
            valid_predictions = model.transform(valid_data)
            validation_auc = evaluator.evaluate(valid_predictions)

            # Calculate training predictions for accuracy
            train_predictions = model.transform(train_data_balanced)
            train_accuracy = train_predictions.filter(train_predictions.label == train_predictions.prediction).count() / float(train_predictions.count())

            # Print current parameters and validation AUC
            print(f"MaxIter: {max_iter}, RegParam: {reg_param}, Training Accuracy: {train_accuracy}, Validation AUC: {validation_auc}")

            # Check if this is the best model
            if validation_auc > best_auc:
                best_auc = validation_auc
                best_model = model
                best_params = {"maxIter": max_iter, "regParam": reg_param}

    return best_model, best_params, best_auc

# Call the function to tune the SVM model
best_model, best_params, best_auc = tune_svm_model(train_data_balanced, valid_data, param_grid, label_col="label")

# Print the best parameters and validation AUC
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary metrics
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxIter: 10, RegParam: 0.1, Training Accuracy: 0.837378554523422, Validation AUC: 0.9015878205698226
MaxIter: 10, RegParam: 0.01, Training Accuracy: 0.843298910455505, Validation AUC: 0.904894810461028
MaxIter: 20, RegParam: 0.1, Training Accuracy: 0.8491624082273758, Validation AUC: 0.9064835286305206
MaxIter: 20, RegParam: 0.01, Training Accuracy: 0.8653527693477658, Validation AUC: 0.9081283634015367
Best Parameters: {'maxIter': 20, 'regParam': 0.01}
Best Validation AUC: 0.9081283634015367
Test AUC: 0.8003580701303888
Test Accuracy: 0.9103629579375848


In [22]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 9592.0         FP: 422.0
Actual: 1     FN: 635.0         TP: 1143.0
[[9592.  422.]
 [ 635. 1143.]]

Metrics:
f1_1:  0.6838169309003889
f1_0:  0.9477792599179882
precision_1:  0.7303514376996805
precision_0:  0.9379094553632541
recall_1:  0.6428571428571429
recall_0:  0.9578589974036349
auc:  0.8003580701303888
accuracy:  0.9103629579375848
sensitivity:  0.6428571428571429
specificity:  0.9578589974036349


In [10]:
##########################Randomforest############################################################
from pyspark.sql import functions as F
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

# Initialize the RandomForestClassifier
rf = RandomForestClassifier(labelCol="label", featuresCol="features_scaled")

# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "numTrees": [10, 20]
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for max_depth in paramGrid["maxDepth"]:
    for num_trees in paramGrid["numTrees"]:
        # Set hyperparameters
        rf.setMaxDepth(max_depth)
        rf.setNumTrees(num_trees)
        
        # Fit the model on the training data
        model = rf.fit(train_data_balanced)
        
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data_balanced)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxDepth: {max_depth}, NumTrees: {num_trees}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxDepth": max_depth, "numTrees": num_trees}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, NumTrees: 10
Train AUC: 0.8232133704544119, Train Accuracy: 0.7527063503338542
Validation AUC: 0.8204301912635296, Validation Accuracy: 0.7935508902013191
MaxDepth: 5, NumTrees: 20
Train AUC: 0.8369765939203302, Train Accuracy: 0.7610384997869015
Validation AUC: 0.8332577715093554, Validation Accuracy: 0.8139845669698668
MaxDepth: 10, NumTrees: 10
Train AUC: 0.8558411897294336, Train Accuracy: 0.7797840602358289
Validation AUC: 0.8461465879992587, Validation Accuracy: 0.8116997887657886
MaxDepth: 10, NumTrees: 20
Train AUC: 0.8633870254293301, Train Accuracy: 0.7821423497655917
Validation AUC: 0.8531758572284617, Validation Accuracy: 0.8154933827650127
Best Parameters: {'maxDepth': 10, 'numTrees': 20}
Best Validation AUC: 0.8531758572284617
Best Validation Accuracy: 0.8154933827650127
Test AUC: 0.6697578154735979
Test Accuracy: 0.814196065128901


In [11]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 8489.0         FP: 453.0
Actual: 1     FN: 1738.0         TP: 1112.0
[[8489.  453.]
 [1738. 1112.]]

Metrics:
f1_1:  0.5037372593431484
f1_0:  0.885700871198289
precision_1:  0.7105431309904153
precision_0:  0.8300576904273003
recall_1:  0.39017543859649123
recall_0:  0.9493401923507045
auc:  0.6697578154735979
accuracy:  0.814196065128901
sensitivity:  0.39017543859649123
specificity:  0.9493401923507045


In [12]:
######################################DecisionTree###############################################################
from pyspark.sql import functions as F
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

# Initialize the DecisionTreeClassifier
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features_scaled")

# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "maxDepth": [5, 10],
    "maxBins": [32, 64]  # Adding maxBins for tuning
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for max_depth in paramGrid["maxDepth"]:
    for max_bins in paramGrid["maxBins"]:
        # Set hyperparameters
        dt.setMaxDepth(max_depth)
        dt.setMaxBins(max_bins)
        
        # Fit the model on the training data
        model = dt.fit(train_data_balanced)
        
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data_balanced)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxDepth: {max_depth}, MaxBins: {max_bins}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxDepth": max_depth, "maxBins": max_bins}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

MaxDepth: 5, MaxBins: 32
Train AUC: 0.5177880825805325, Train Accuracy: 0.7526140076715443
Validation AUC: 0.5150327582323603, Validation Accuracy: 0.7710479803422856
MaxDepth: 5, MaxBins: 64
Train AUC: 0.5177949303527686, Train Accuracy: 0.7526353175166927
Validation AUC: 0.5150532617359078, Validation Accuracy: 0.7710910893650041
MaxDepth: 10, MaxBins: 32
Train AUC: 0.23227842304437524, Train Accuracy: 0.7899062366813467
Validation AUC: 0.23103850619468436, Validation Accuracy: 0.8248911497176359
MaxDepth: 10, MaxBins: 64
Train AUC: 0.232018084181572, Train Accuracy: 0.7904034664014775
Validation AUC: 0.23034026507447997, Validation Accuracy: 0.828210544466957
Best Parameters: {'maxDepth': 5, 'maxBins': 64}
Best Validation AUC: 0.5150532617359078
Best Validation Accuracy: 0.7710910893650041
Test AUC: 0.6405018153562102
Test Accuracy: 0.7776458616010855


In [13]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 8081.0         FP: 476.0
Actual: 1     FN: 2146.0         TP: 1089.0
[[8081.  476.]
 [2146. 1089.]]

Metrics:
f1_1:  0.45375
f1_0:  0.8604131175468483
precision_1:  0.6958466453674121
precision_0:  0.7901632932433754
recall_1:  0.3366306027820711
recall_0:  0.9443730279303494
auc:  0.6405018153562102
accuracy:  0.7776458616010855
sensitivity:  0.3366306027820711
specificity:  0.9443730279303494


In [9]:
#######################################MLP############################################################################
from pyspark.sql import functions as 
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

# Define the Multilayer Perceptron structure
input_features = train_data_balanced.select("features_scaled").first()[0].size
layers = [input_features, 128, 2]  # Adjusted for your high-dimensional input

# Initialize the MultilayerPerceptronClassifier
mlp = MultilayerPerceptronClassifier(
    labelCol="label",
    featuresCol="features_scaled",
    layers=layers,
    blockSize=128,  # Adjusted block size for memory efficiency
    seed=1234,
    solver="gd",  # Stochastic Gradient Descent
    maxIter=50,  # Reduced maximum iterations to prevent overfitting and improve memory efficiency
    stepSize=0.01,  # Default step size for convergence
    tol=1e-5  # Tolerance for convergence
)

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Hyperparameter grid for tuning (optional, since we've reduced maxIter to 50 for efficiency)
paramGrid = {
    "maxIter": [50],  # We reduce maxIter to 50 to prevent memory issues
    "stepSize": [0.01]  # We leave stepSize as 0.01 for simplicity
}

# Loop through all combinations of hyperparameters
for max_iter in paramGrid["maxIter"]:
    for step_size in paramGrid["stepSize"]:
        # Set hyperparameters
        mlp.setMaxIter(max_iter)
        mlp.setStepSize(step_size)
        
        # Fit the model on the training data
        model = mlp.fit(train_data_balanced)
        
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data_balanced)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxIter: {max_iter}, StepSize: {step_size}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxIter": max_iter, "stepSize": step_size}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Test Accuracy: 0.7431309362279511


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 7697.0         FP: 499.0
Actual: 1     FN: 2530.0         TP: 1066.0
[[7697.  499.]
 [2530. 1066.]]

Metrics:
f1_1:  0.41309823677581864
f1_0:  0.835585952342181
precision_1:  0.681150159744409
precision_0:  0.7526156253055637
recall_1:  0.296440489432703
recall_0:  0.9391166422645193
auc:  0.6177785658486111
accuracy:  0.7431309362279511
sensitivity:  0.296440489432703
specificity:  0.9391166422645193


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import numpy as np
# setting random seed for notebook reproducability
rnd_seed=23
np.random.seed=rnd_seed
np.random.set_state=rnd_seed
train_data, valid_data, test_data = balanced_df.randomSplit([.7,.2,.1], seed=rnd_seed)

In [4]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [5]:
#Step6
###########################Included Standard Scalar #############################################################
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint, StandardScaler

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and return size hint stages for the pipeline
def get_vector_size_hint_stage(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 0.01% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")
        # Return a VectorSizeHint stage for the pipeline if vector size is valid
        if vector_size > 0:
            return VectorSizeHint(inputCol=col_name, size=vector_size)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")
    
    return None

# Step 3: Create a list of VectorSizeHint stages for each vector column
vector_size_hint_stages = []
for col_name in vector_cols:
    print(f"Getting VectorSizeHint for vector column: '{col_name}'")
    size_hint_stage = get_vector_size_hint_stage(train_data, col_name)
    if size_hint_stage:
        vector_size_hint_stages.append(size_hint_stage)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")

# Step 6: Initialize StandardScaler
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Step 7: Create a pipeline with VectorSizeHint stages, VectorAssembler, and StandardScaler
pipeline_stages = vector_size_hint_stages + [final_assembler, standardScaler]
pipeline_final = Pipeline(stages=pipeline_stages)

# Step 8: Fit the pipeline on train_data
model_final = pipeline_final.fit(train_data)
print("Pipeline fitting done.")

# Step 9: Transform train, valid, and test datasets using the fitted pipeline
train_data = model_final.transform(train_data)
valid_data = model_final.transform(valid_data)
test_data = model_final.transform(test_data)
print("Pipeline transformation done.")

Getting VectorSizeHint for vector column: 'gender_onehot'
Column 'gender_onehot' vector size: 4
Getting VectorSizeHint for vector column: 'race_onehot'
Column 'race_onehot' vector size: 7
Final input columns for feature assembly: ['gender_onehot', 'race_onehot', 'age_of_TBI_diagnosis', 'MedicalHistory', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B

Pipeline fitting done.
Pipeline transformation done.


In [6]:
#Step7
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [7]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Check if we need to upsample or downsample
if count_zeros > count_ones:
    # Upsample the minority class
    upsample_ratio = count_zeros // count_ones  # Calculate integer part of ratio
    remaining_samples_fraction = (count_zeros % count_ones) / count_ones  # Remaining fraction

    # Duplicate the minority class to match the majority class count
    upsampled_minority_class_df = minority_class_df
    for _ in range(upsample_ratio - 1):  # -1 because we already have one instance
        upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)

    # Add the remaining samples to reach exact count
    upsampled_minority_class_df = upsampled_minority_class_df.union(
        minority_class_df.sample(withReplacement=True, fraction=remaining_samples_fraction)
    )
    
    # Combine with majority class
    train_data_balanced = majority_class_df.union(upsampled_minority_class_df)

else:
    # Downsample the majority class if count_ones > count_zeros
    downsample_fraction = count_ones / count_zeros
    downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)
    
    # Combine with minority class
    train_data_balanced = downsampled_majority_class_df.union(minority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame: 70414
Number of 1's in the balanced DataFrame: 70451


In [12]:
####################################### Final Improved - MLP ##############################################
from pyspark.sql import functions as F
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

# Define the Multilayer Perceptron structure
input_features = train_data_balanced.select("features_scaled").first()[0].size
layers = [input_features, 128, 2]  # Adjusted for your high-dimensional input

# Initialize the MultilayerPerceptronClassifier
mlp = MultilayerPerceptronClassifier(
    labelCol="label",
    featuresCol="features_scaled",
    layers=layers,
    blockSize=128,  # Adjusted block size for memory efficiency
    seed=1234,
    solver="gd",  # Stochastic Gradient Descent
    maxIter=50,  # Reduced maximum iterations to prevent overfitting and improve memory efficiency
    stepSize=0.01,  # Default step size for convergence
    tol=1e-5  # Tolerance for convergence
)

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Hyperparameter grid for tuning (optional, since we've reduced maxIter to 50 for efficiency)
paramGrid = {
    "maxIter": [50],  # We reduce maxIter to 50 to prevent memory issues
    "stepSize": [0.01]  # We leave stepSize as 0.01 for simplicity
}

# Loop through all combinations of hyperparameters
for max_iter in paramGrid["maxIter"]:
    for step_size in paramGrid["stepSize"]:
        # Set hyperparameters
        mlp.setMaxIter(max_iter)
        mlp.setStepSize(step_size)
        
        # Fit the model on the training data
        model = mlp.fit(train_data_balanced)
        
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data_balanced)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"MaxIter: {max_iter}, StepSize: {step_size}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"maxIter": max_iter, "stepSize": step_size}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

# Adjust the decision threshold to improve sensitivity
threshold = 0.3  # Decrease the threshold to be more sensitive to class 1
adjusted_predictions = test_predictions.withColumn(
    "adjusted_prediction",
    F.when(test_predictions["rawPrediction"].getItem(1) > threshold, 1).otherwise(0)
)

# Evaluate the model with the adjusted threshold
adjusted_predictionAndTarget = adjusted_predictions.select("label", "adjusted_prediction")
adjusted_predictionAndTarget_rdd = adjusted_predictionAndTarget.rdd.map(tuple)
metrics_multi_adjusted = MulticlassMetrics(adjusted_predictionAndTarget_rdd)

# Print the adjusted metrics
adjusted_accuracy = metrics_multi_adjusted.accuracy
adjusted_sensitivity = metrics_multi_adjusted.recall(1)  # Sensitivity (Recall for class 1)
print(f"Adjusted Test Accuracy: {adjusted_accuracy}")
print(f"Adjusted Sensitivity (Recall for class 1): {adjusted_sensitivity}")

MaxIter: 50, StepSize: 0.01
Train AUC: 0.7043940568622564, Train Accuracy: 0.7147197671529478
Validation AUC: 0.6998155023070921, Validation Accuracy: 0.7488468336422813
Best Parameters: {'maxIter': 50, 'stepSize': 0.01}
Best Validation AUC: 0.6998155023070921
Best Validation Accuracy: 0.7488468336422813
Test AUC: 0.6142449684651694
Test Accuracy: 0.7472014925373134


AnalysisException: "Can't extract value from rawPrediction#1632266: need struct type but got struct<type:tinyint,size:int,indices:array<int>,values:array<double>>;"

In [ ]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

In [11]:
################################LogRegression ###########################################################
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

# Initialize the Logistic Regression model
lr = LogisticRegression(labelCol="label", featuresCol="features_scaled", maxIter=10)

# Set up the parameter grid for hyperparameter tuning
paramGrid = {
    "regParam": [0.01, 0.1],  # Regularization parameter
    "elasticNetParam": [0.0, 0.5, 1.0]  # ElasticNet mixing parameter
}

# Use BinaryClassificationEvaluator for model evaluation with AUC metric
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Store the best model and the best AUC score
best_model = None
best_auc = 0.0
best_accuracy = 0.0
best_params = {}

# Loop through all combinations of hyperparameters
for reg_param in paramGrid["regParam"]:
    for elastic_net_param in paramGrid["elasticNetParam"]:
        # Set hyperparameters
        lr.setRegParam(reg_param)
        lr.setElasticNetParam(elastic_net_param)
        
        # Fit the model on the training data
        model = lr.fit(train_data_balanced)
        
        # *** Train data evaluation ***
        train_predictions = model.transform(train_data_balanced)

        # Calculate train AUC
        train_auc = evaluator.evaluate(train_predictions)

        # Calculate train accuracy
        predictionAndTarget_train = train_predictions.select("label", "prediction")
        predictionAndTarget_train_rdd = predictionAndTarget_train.rdd.map(tuple)
        metrics_multi_train = MulticlassMetrics(predictionAndTarget_train_rdd)
        train_accuracy = metrics_multi_train.accuracy        

        # Evaluate the model on the validation data
        valid_predictions = model.transform(valid_data)

        # Calculate validation AUC
        validation_auc = evaluator.evaluate(valid_predictions)

        # Calculate validation accuracy
        predictionAndTarget_valid = valid_predictions.select("label", "prediction")
        predictionAndTarget_valid_rdd = predictionAndTarget_valid.rdd.map(tuple)
        metrics_multi = MulticlassMetrics(predictionAndTarget_valid_rdd)
        validation_accuracy = metrics_multi.accuracy

        # Print the current parameters, validation AUC, and validation accuracy
        print(f"RegParam: {reg_param}, ElasticNetParam: {elastic_net_param}")
        print(f"Train AUC: {train_auc}, Train Accuracy: {train_accuracy}")
        print(f"Validation AUC: {validation_auc}, Validation Accuracy: {validation_accuracy}")
        
        # Check if this is the best model
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_accuracy = validation_accuracy
            best_model = model
            best_params = {"regParam": reg_param, "elasticNetParam": elastic_net_param}

# Print the best parameters and validation AUC and Accuracy
print(f"Best Parameters: {best_params}")
print(f"Best Validation AUC: {best_auc}")
print(f"Best Validation Accuracy: {best_accuracy}")

# Make predictions on the test data using the best model found
test_predictions = best_model.transform(test_data)

# Prepare for evaluation by selecting label and prediction columns
predictionAndTarget = test_predictions.select("label", "prediction")
predictionAndTarget = predictionAndTarget.withColumn('label', predictionAndTarget.label.cast('double'))

# Convert the DataFrame to an RDD and compute the metrics
predictionAndTarget_rdd = predictionAndTarget.rdd.map(tuple)

# Binary and Multiclass metrics
metrics_binary = BinaryClassificationMetrics(predictionAndTarget_rdd)
metrics_multi = MulticlassMetrics(predictionAndTarget_rdd)

# Example: Print AUC and Accuracy
print(f"Test AUC: {metrics_binary.areaUnderROC}")
print(f"Test Accuracy: {metrics_multi.accuracy}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Test Accuracy: 0.9012042062415196


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
#Step11
###################Using Balanced Samples with Single Validation Set& multiple param ###############################################
confusion_matrix = metrics_multi.confusionMatrix().toArray()
tn, fp, fn, tp = confusion_matrix[0][0], confusion_matrix[0][1], confusion_matrix[1][0], confusion_matrix[1][1]
sensitivity = tp / (fn + tp)
specificity = tn / (tn + fp)

# Compute other metrics
accuracy = metrics_multi.accuracy
f1_1 = metrics_multi.fMeasure(1.0)
f1_0 = metrics_multi.fMeasure(0.0)
precision_1 = metrics_multi.precision(1.0)
precision_0 = metrics_multi.precision(0.0)
recall_1 = metrics_multi.recall(1.0)
recall_0 = metrics_multi.recall(0.0)
auc = metrics_binary.areaUnderROC
# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"          Predicted: 0     Predicted: 1")
print(f"Actual: 0     TN: {tn}         FP: {fp}")
print(f"Actual: 1     FN: {fn}         TP: {tp}")
print(confusion_matrix)
# Print the metrics
print("\nMetrics:")
print("f1_1: ", f1_1)
print("f1_0: ", f1_0)
print("precision_1: ", precision_1)
print("precision_0: ", precision_0)
print("recall_1: ", recall_1)
print("recall_0: ", recall_0)
print("auc: ", auc)
print("accuracy: ", accuracy)
print('sensitivity: ', sensitivity)
print('specificity: ', specificity)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Confusion Matrix:
          Predicted: 0     Predicted: 1
Actual: 0     TN: 9472.0         FP: 410.0
Actual: 1     FN: 755.0         TP: 1155.0
[[9472.  410.]
 [ 755. 1155.]]

Metrics:
f1_1:  0.6647482014388489
f1_0:  0.942065741707693
precision_1:  0.7380191693290735
precision_0:  0.9261758091326879
recall_1:  0.6047120418848168
recall_0:  0.9585104229912973
auc:  0.781611232438057
accuracy:  0.9012042062415196
sensitivity:  0.6047120418848168
specificity:  0.9585104229912973


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>